# NPC2 Proteomics analysis (corrected for covariates)
Simple analysis for NPC2 protein is on "NPC2_proteomics_prep.ipynb"

## 1. Data prep - NPC2 protein

In [ ]:
# Rescuing patient information
import os
import pandas as pd
import numpy as np

NPC2_splice_donor_proteo = pd.read_csv("/mnt/project/NPC2_splice_donor_proteo.txt", sep="\t")
NPC2_stop_gain_proteo = pd.read_csv("/mnt/project/NPC2_stop_gain_proteo.txt", sep="\t")
NPC2_D91N_proteo = pd.read_csv("/mnt/project/NPC2_D91N_proteo.txt", sep="\t")
No_NPC2_proteo = pd.read_csv("/mnt/project/No_NPC2_proteo.txt", sep="\t")

NPC2_splice_donor_proteo.head()

In [ ]:
# Count HET (1) and HOM (0) per variant type
NPC2_splice_donor_proteo["COUNT"].value_counts()

In [ ]:
NPC2_stop_gain_proteo["COUNT"].value_counts()

In [ ]:
NPC2_D91N_proteo["COUNT"].value_counts()

In [ ]:
# Rescuing protein abundances
proteomics = pd.read_csv("/mnt/project/proteomics_results/complete_proteomics_df.txt", sep="\t") # also in DNAnexus:proteomics_results
proteomics.head(5)

In [ ]:
# Look for NPC2 protein
proteomics.columns = proteomics.columns.str.replace("olink_instance_0.", "", case=False)

matches2 = [col for col in proteomics.columns if col.lower().startswith("npc2")]
matches2

In [ ]:
splice_HET = NPC2_splice_donor_proteo[NPC2_splice_donor_proteo["COUNT"] == 1]
splice_HOM = NPC2_splice_donor_proteo[NPC2_splice_donor_proteo["COUNT"] == 0]

NPC2_splice_donor_HET_abund = proteomics[proteomics['eid'].isin(splice_HET['IID'])]
NPC2_splice_donor_HOM_abund = proteomics[proteomics['eid'].isin(splice_HOM['IID'])]
NPC2_stop_gain_abund = proteomics[proteomics['eid'].isin(NPC2_stop_gain_proteo['IID'])]
NPC2_D91N_abund = proteomics[proteomics['eid'].isin(NPC2_D91N_proteo['IID'])]
No_NPC2_abund = proteomics[proteomics['eid'].isin(No_NPC2_proteo['IID'])]

In [ ]:
len(NPC2_splice_donor_HET_abund)

In [ ]:
len(NPC2_splice_donor_HOM_abund)

In [ ]:
len(NPC2_stop_gain_abund)

In [ ]:
len(NPC2_D91N_abund)

In [ ]:
len(No_NPC2_abund)

In [ ]:
NPC2_splice_donor_HET_abund_cts = NPC2_splice_donor_HET_abund[ ["eid"] + matches2 ]
NPC2_splice_donor_HOM_abund_cts = NPC2_splice_donor_HOM_abund[ ["eid"] + matches2 ]
NPC2_stop_gain_abund_cts = NPC2_stop_gain_abund[ ["eid"] + matches2 ]
NPC2_D91N_abund_cts = NPC2_D91N_abund[ ["eid"] + matches2 ]
No_NPC2_abund_cts = No_NPC2_abund[ ["eid"] + matches2 ]

NPC2_splice_donor_HET_abund_cts.head()

In [ ]:
# Merge metadata information (age at baseline and sex)
metadata = pd.read_csv("/mnt/project/proteomics_results/age_sex_proteomics_df.txt", sep="\t") # also in DNAnexus:proteomics_results
metadata.head(5)

In [ ]:
NPC2_splice_donor_HET_abund_cts = metadata.merge(NPC2_splice_donor_HET_abund_cts, left_on="participant.eid", right_on="eid")
NPC2_splice_donor_HOM_abund_cts = metadata.merge(NPC2_splice_donor_HOM_abund_cts, left_on="participant.eid", right_on="eid")
NPC2_stop_gain_abund_cts = metadata.merge(NPC2_stop_gain_abund_cts, left_on="participant.eid", right_on="eid")
NPC2_D91N_abund_cts = metadata.merge(NPC2_D91N_abund_cts, left_on="participant.eid", right_on="eid")
No_NPC2_abund_cts = metadata.merge(No_NPC2_abund_cts, left_on="participant.eid", right_on="eid")

NPC2_splice_donor_HET_abund_cts.head()

In [ ]:
len(NPC2_splice_donor_HET_abund_cts)

In [ ]:
len(NPC2_splice_donor_HOM_abund_cts)

In [ ]:
len(NPC2_stop_gain_abund_cts)

In [ ]:
len(NPC2_D91N_abund_cts)

In [ ]:
len(No_NPC2_abund_cts)

## 2. Differential expression, corrected for covariates
Used multiple linear regression, only with NPC2 protein

In [ ]:
!pip install statsmodels

In [ ]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

# --------------------------------------------------------
# 1. Build a unified dataframe with covariates + proteins
# --------------------------------------------------------

# Add phenotype columns to your Case and Control dataframes
# NPC2_splice_donor_HET_abund_cts["case_status"]  = 1
# NPC2_splice_donor_HOM_abund_cts["case_status"]  = 1
# NPC2_stop_gain_abund_cts["case_status"]  = 1
NPC2_D91N_abund_cts["case_status"]  = 1
No_NPC2_abund_cts["case_status"] = 0

# Make single combined dataframe
df = pd.concat([NPC2_D91N_abund_cts, No_NPC2_abund_cts], ignore_index=True)

# Standardize column names
df = df.rename(columns={
    "participant.p21003_i0": "age",
    "participant.p31": "sex",
})

# Identify protein columns
protein_cols = [c for c in df.columns 
                if c not in ["participant.eid", "eid", "case_status", "age", "sex"]]

df.head()

In [ ]:
# --------------------------------------------------------
# 2. Run regression for each protein
# --------------------------------------------------------

results_list = []

for protein in protein_cols:
    tmp = df[["case_status", "age", "sex", protein]].dropna()

    # y = protein
    y = tmp[protein]

    # X = case_status + age + sex
    X = tmp[["case_status", "age", "sex"]]
    X = sm.add_constant(X)

    # OLS
    model = sm.OLS(y, X).fit()

    # Extract results for the case_status coefficient
    beta = model.params["case_status"]
    pval = model.pvalues["case_status"]

    # Compute means
    mean_case = tmp.loc[tmp.case_status == 1, protein].mean()
    mean_control = tmp.loc[tmp.case_status == 0, protein].mean()

    results_list.append([protein, mean_case, mean_control, beta, pval])

# Convert to dataframe
results = pd.DataFrame(results_list, columns=[
    "protein_name", "mean_case", "mean_control", "beta_case_status", "p_value"
])

In [ ]:
# --------------------------------------------------------
# 3. FDR correction (only when more than 1 protein being tested
# --------------------------------------------------------

#results["FDR_p_value"] = multipletests(results["p_value"], method="fdr_bh")[1]

In [ ]:
# Sort by FDR
#results = results.sort_values("p_value")

# Show results
results

In [ ]:
# Upload final files
results.to_csv("regression_NPC2_protein_D91N_vs_No-NPC2_age-sex_corrected.txt", sep="\t", index=False)
!dx upload regression_NPC2_protein_D91N_vs_No-NPC2_age-sex_corrected.txt --destination proteomics_results/

In [ ]:
# Plot boxplots of selected proteins
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# proteins you want to plot (must be in protein_cols)
selected_proteins = ["npc2"]   # or e.g. ["npc2", "ctsb", "ctsd", ...]

plot_rows = []

for protein in selected_proteins:
    # subset with all columns we need
    tmp = df[["case_status", "age", "sex", protein]].dropna().copy()

    # regress protein ~ age + sex  (NO case_status here)
    X_cov = sm.add_constant(tmp[["age", "sex"]])
    y = tmp[protein]
    model_cov = sm.OLS(y, X_cov).fit()

    # covariate-adjusted abundance: residuals + intercept
    tmp["adj_abundance"] = model_cov.resid + model_cov.params["const"]

    # add labels for plotting
    tmp["group"] = np.where(tmp["case_status"] == 1, "case", "control")
    tmp["protein"] = protein

    plot_rows.append(tmp[["group", "protein", "adj_abundance"]])

# combined long-format dataframe for plotting
plot_df = pd.concat(plot_rows, ignore_index=True)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 8))

sns.boxplot(
    data=plot_df,
    x="group", y="adj_abundance",
    order=["control", "case"],
    palette="Set2",
    ax=ax,
)

sns.stripplot(
    data=plot_df,
    x="group", y="adj_abundance",
    order=["control", "case"],
    color="black",
    alpha=0.4,
    dodge=False,
    jitter=True,
    ax=ax,
)

ax.set_title("Covariate-adjusted abundance (age, sex): case vs control")
ax.set_xlabel("Group")
ax.set_ylabel("Adjusted abundance")

fig.tight_layout()
fig.savefig("/tmp/NPC2_boxplots_NPC2_D91N_vs_No-NPC_age-sex_corrected.pdf",
            format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Upload final files
import os

outdir = "/tmp"
#txt_path = os.path.join(outdir, "t-test_NPC2_protein_NPC2_D91N_vs_No-NPC.txt")

#results.to_csv(txt_path, sep="\t", index=False)

#!dx upload /tmp/t-test_NPC2_protein_NPC2_D91N_vs_No-NPC.txt --destination proteomics_results/
!dx upload /tmp/NPC2_boxplots_NPC2_D91N_vs_No-NPC_age-sex_corrected.pdf --destination proteomics_results/